# 97 — Cluster and review nodal stack source positions (SAFE, v6)

Notebook 97 consumes notebook 96's manifests and separates the interpretation
workflow into three explicit stages.

## Stage 1 — Canonical source clustering

Stacks are processed in integration-priority order. Each stack is assigned to
the nearest existing cluster on the same line when its source position is within
the configured tolerance. Otherwise, it starts a new cluster.

The cluster's canonical source position and canonical stack are defined by its
highest-authority member: the lowest numerical `priority`, followed by stable
source-position and stack-ID ordering.

## Stage 2 — Comparison-task generation

Every multi-member cluster generates all unique stack pairs. This remains
branch-agnostic, so future product branches can be added through notebook 96.

## Stage 3 — Waveform evidence

Waveforms are compared using:

- a configurable arrival-focused relative-time window;
- identical preprocessing and bandpass filtering;
- receiver-level **envelope** cross-correlation over a broad lag range;
- a robust gather-wide coarse lag estimate from those envelopes;
- a second phase-sensitive correlation constrained around that coarse lag;
- waveform, envelope, polarity, SNR, and lag-coherence diagnostics.

The automatic result is deliberately conservative:

- `waveform_match_supported` means strong positive evidence;
- `waveform_match_inconclusive` means the available waveform evidence is
  insufficient or inconsistent;
- automatic waveform screening never rejects a geometry-based source cluster.

This notebook does not merge, overwrite, or resave stack waveforms.


## 1. Configuration

In [1]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from obspy import read
from obspy.signal.cross_correlation import correlate, xcorr_max
from scipy.signal import hilbert

PROJECT_ROOT = Path('/Volumes/tachyon/LBSSP_DATA')
MANIFEST_ROOT = PROJECT_ROOT / '96_unified_nodal_stack_manifest'
OUT_ROOT = PROJECT_ROOT / '97_nodal_source_cluster_review'
FIGURE_ROOT = OUT_ROOT / 'figures'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

STACK_MANIFEST_PATH = MANIFEST_ROOT / '96_nodal_stack_manifest.csv'
FILE_MANIFEST_PATH = MANIFEST_ROOT / '96_nodal_stack_file_manifest.csv'

SOURCE_TOLERANCE_M = 0.25
RECEIVER_TOLERANCE_M = 0.25
COMPONENT = 'Z'

# Arrival-focused comparison window, in seconds from each trace start.
# Adjust after inspecting representative stack figures.
ANALYSIS_START_S = 0.00
ANALYSIS_END_S = 1.25

# Identical preprocessing for both traces.
FILTER_FREQMIN_HZ = 15.0
FILTER_FREQMAX_HZ = 120.0
FILTER_CORNERS = 4
FILTER_ZEROPHASE = True
TAPER_FRACTION = 0.02

# Coarse receiver-level envelope-lag search. This must be wider than\n# any plausible difference between independently generated stack origins.\nCOARSE_MAX_LAG_S = 0.40
COARSE_MAX_LAG_S = 0.50
# Robust gather-wide lag estimation.
MIN_COARSE_ENVELOPE_CORRELATION_FOR_LAG = 0.25
MAX_COARSE_LAG_DEVIATION_S = 0.08
COHERENT_LAG_HALF_WIDTH_S = 0.025
MIN_RECEIVERS_FOR_GATHER_LAG = 3

# Trace quality controls.
MIN_ANALYSIS_DURATION_S = 0.10
MIN_SNR = 1.5
# When the analysis window begins at trace time zero, no pre-signal
# noise window exists. Missing SNR is then treated as unavailable,
# not as an automatic failure.
ALLOW_MISSING_SNR = True
NOISE_WINDOW_DURATION_S = 0.10
MIN_COMMON_RECEIVERS_FOR_GATHER = 3

# Positive-evidence thresholds.
TRACE_ACCEPT_CORRELATION = 0.70
TRACE_ACCEPT_ENVELOPE_CORRELATION = 0.70
GATHER_ACCEPT_MEDIAN_CORRELATION = 0.70
GATHER_ACCEPT_MEDIAN_ENVELOPE_CORRELATION = 0.70
GATHER_ACCEPT_FRACTION = 0.60
GATHER_ACCEPT_COHERENT_LAG_FRACTION = 0.60
GATHER_MAX_LAG_MAD_S = 0.010

MAKE_REVIEW_FIGURES = True
MAX_FIGURES = None

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 280)

print('Manifest root:', MANIFEST_ROOT)
print('Output root:', OUT_ROOT)
print('Source tolerance:', SOURCE_TOLERANCE_M, 'm')
print('Receiver tolerance:', RECEIVER_TOLERANCE_M, 'm')
print('Component:', COMPONENT)
print('Analysis window:', ANALYSIS_START_S, 'to', ANALYSIS_END_S, 's')
print('Filter:', FILTER_FREQMIN_HZ, 'to', FILTER_FREQMAX_HZ, 'Hz')

print('Coarse envelope lag search: +/-', COARSE_MAX_LAG_S, 's')

Manifest root: /Volumes/tachyon/LBSSP_DATA/96_unified_nodal_stack_manifest
Output root: /Volumes/tachyon/LBSSP_DATA/97_nodal_source_cluster_review
Source tolerance: 0.25 m
Receiver tolerance: 0.25 m
Component: Z
Analysis window: 0.0 to 1.25 s
Filter: 15.0 to 120.0 Hz
Coarse envelope lag search: +/- 0.5 s


## 2. Load and validate notebook-96 manifests

In [2]:
for path in [STACK_MANIFEST_PATH, FILE_MANIFEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f'Missing notebook-96 output: {path}\n'
            'Run notebook 96 first.'
        )

stacks = pd.read_csv(STACK_MANIFEST_PATH, low_memory=False)
files = pd.read_csv(FILE_MANIFEST_PATH, low_memory=False)

required_stack_columns = {
    'stack_id', 'catalog_branch', 'line', 'source_x_m',
    'priority', 'merge_stage',
}
missing_stack_columns = sorted(required_stack_columns - set(stacks.columns))
if missing_stack_columns:
    raise RuntimeError(
        'Notebook-96 stack manifest lacks required columns: '
        f'{missing_stack_columns}'
    )

stacks['source_x_m'] = pd.to_numeric(stacks['source_x_m'], errors='coerce')
stacks['priority'] = pd.to_numeric(stacks['priority'], errors='coerce')

files['component'] = files['component'].astype(str).str.upper()
files['file_type'] = files['file_type'].astype(str).str.lower()
files['file_exists'] = (
    files['file_exists']
    .fillna(False)
    .astype(str)
    .str.lower()
    .isin(['true', '1', 'yes'])
)

invalid = stacks.loc[
    stacks.source_x_m.isna() | stacks.priority.isna(),
    ['stack_id', 'catalog_branch', 'line', 'source_x_m', 'priority'],
]
if len(invalid):
    display(invalid)
    raise RuntimeError(
        'All stacks require finite source_x_m and priority before clustering.'
    )

print('Stack rows:', len(stacks))
print('File rows:', len(files))
display(
    stacks.groupby(
        ['catalog_branch', 'priority', 'merge_stage'],
        dropna=False,
    ).size().reset_index(name='n_stacks')
)

Stack rows: 238
File rows: 978


,catalog_branch,priority,merge_stage,n_stacks
0,geode_linked,1,reference,78
1,geode_linked,2,secondary_geode_extension,36
2,geode_linked,3,streamer_extension,80
3,nodal_only,4,candidate_extension,44


## 3. Assign canonical source clusters

In [3]:
def make_cluster_id(line, number):
    safe_line = ''.join(
        character if character.isalnum() else '_'
        for character in str(line)
    )
    return f'{safe_line}_SRC_{number:04d}'


def choose_canonical_member(member_rows):
    ordered = member_rows.sort_values(
        ['priority', 'source_x_m', 'stack_id'],
        kind='stable',
    )
    return ordered.iloc[0]


cluster_membership_rows = []
cluster_summary_rows = []

for line, line_stacks in stacks.groupby('line', sort=True, dropna=False):
    ordered = line_stacks.sort_values(
        ['priority', 'source_x_m', 'stack_id'],
        kind='stable',
    ).reset_index(drop=True)

    clusters = []

    for stack in ordered.itertuples(index=False):
        possible = []
        for cluster_index, cluster in enumerate(clusters):
            distance = abs(
                float(stack.source_x_m)
                - float(cluster['canonical_source_x_m'])
            )
            if distance <= SOURCE_TOLERANCE_M:
                possible.append(
                    (
                        distance,
                        cluster['canonical_priority'],
                        cluster['canonical_stack_id'],
                        cluster_index,
                    )
                )

        if possible:
            _, _, _, cluster_index = min(possible)
            clusters[cluster_index]['members'].append(stack._asdict())
        else:
            clusters.append({
                'members': [stack._asdict()],
                'canonical_source_x_m': float(stack.source_x_m),
                'canonical_priority': int(stack.priority),
                'canonical_stack_id': str(stack.stack_id),
            })

        # Recompute the canonical member after every assignment. A newly added
        # higher-authority branch can therefore become the canonical member.
        chosen_cluster = clusters[cluster_index] if possible else clusters[-1]
        chosen_frame = pd.DataFrame(chosen_cluster['members'])
        canonical = choose_canonical_member(chosen_frame)
        chosen_cluster['canonical_source_x_m'] = float(canonical.source_x_m)
        chosen_cluster['canonical_priority'] = int(canonical.priority)
        chosen_cluster['canonical_stack_id'] = str(canonical.stack_id)

    clusters.sort(
        key=lambda cluster: (
            cluster['canonical_source_x_m'],
            cluster['canonical_priority'],
            cluster['canonical_stack_id'],
        )
    )

    for cluster_number, cluster in enumerate(clusters, start=1):
        cluster_id = make_cluster_id(line, cluster_number)
        members = pd.DataFrame(cluster['members'])
        canonical = choose_canonical_member(members)

        source_offsets = (
            members.source_x_m.astype(float) - float(canonical.source_x_m)
        )

        cluster_summary_rows.append({
            'source_cluster_id': cluster_id,
            'line': line,
            'canonical_stack_id': str(canonical.stack_id),
            'canonical_catalog_branch': canonical.catalog_branch,
            'canonical_priority': int(canonical.priority),
            'canonical_merge_stage': canonical.merge_stage,
            'canonical_source_x_m': float(canonical.source_x_m),
            'n_stack_products': len(members),
            'n_catalog_branches': members.catalog_branch.nunique(),
            'minimum_source_x_m': float(members.source_x_m.min()),
            'maximum_source_x_m': float(members.source_x_m.max()),
            'source_span_m': float(
                members.source_x_m.max() - members.source_x_m.min()
            ),
            'maximum_abs_offset_from_canonical_m': float(
                source_offsets.abs().max()
            ),
            'is_multi_stack_cluster': len(members) > 1,
            'is_cross_branch_cluster': members.catalog_branch.nunique() > 1,
        })

        for member in members.itertuples(index=False):
            cluster_membership_rows.append({
                'source_cluster_id': cluster_id,
                'line': line,
                'stack_id': str(member.stack_id),
                'catalog_branch': member.catalog_branch,
                'priority': int(member.priority),
                'merge_stage': member.merge_stage,
                'source_x_m': float(member.source_x_m),
                'source_offset_from_canonical_m': (
                    float(member.source_x_m) - float(canonical.source_x_m)
                ),
                'is_canonical_stack': str(member.stack_id) == str(canonical.stack_id),
                'canonical_stack_id': str(canonical.stack_id),
                'canonical_source_x_m': float(canonical.source_x_m),
            })

source_clusters = pd.DataFrame(cluster_summary_rows)
cluster_membership = pd.DataFrame(cluster_membership_rows)

if len(source_clusters):
    source_clusters = source_clusters.sort_values(
        ['line', 'canonical_source_x_m', 'source_cluster_id']
    ).reset_index(drop=True)

if len(cluster_membership):
    cluster_membership = cluster_membership.sort_values(
        ['line', 'canonical_source_x_m', 'priority', 'source_x_m', 'stack_id']
    ).reset_index(drop=True)

print('Source clusters:', len(source_clusters))
print(
    'Multi-stack clusters:',
    int(source_clusters.is_multi_stack_cluster.sum()),
)
print(
    'Cross-branch clusters:',
    int(source_clusters.is_cross_branch_cluster.sum()),
)
display(source_clusters.head(30))

Source clusters: 198
Multi-stack clusters: 37
Cross-branch clusters: 18


,source_cluster_id,line,canonical_stack_id,canonical_catalog_branch,canonical_priority,canonical_merge_stage,canonical_source_x_m,n_stack_products,n_catalog_branches,minimum_source_x_m,maximum_source_x_m,source_span_m,maximum_abs_offset_from_canonical_m,is_multi_stack_cluster,is_cross_branch_cluster
0,T1_SRC_0001,T1,NODALONLYSTACK_MAY19_010M_x0010.0m,nodal_only,4,candidate_extension,10.0,1,1,10.0,10.0,0.0,0.0,False,False
1,T1_SRC_0002,T1,NODALONLYSTACK_MAY19_036M_x0036.0m,nodal_only,4,candidate_extension,36.0,1,1,36.0,36.0,0.0,0.0,False,False
2,T1_SRC_0003,T1,NODALSTACK_T1_T1_2m_refraction_F3047_x0043.0m,geode_linked,2,secondary_geode_extension,43.0,1,1,43.0,43.0,0.0,0.0,False,False
3,T1_SRC_0004,T1,NODALSTACK_T1_T1_2m_refraction_F3048_x0047.0m,geode_linked,2,secondary_geode_extension,47.0,1,1,47.0,47.0,0.0,0.0,False,False
4,T1_SRC_0005,T1,NODALSTACK_T1_T1_2m_refraction_F3049_x0051.0m,geode_linked,2,secondary_geode_extension,51.0,1,1,51.0,51.0,0.0,0.0,False,False
5,T1_SRC_0006,T1,NODALSTACK_T1_T1_2m_refraction_F3050_x0055.0m,geode_linked,2,secondary_geode_extension,55.0,1,1,55.0,55.0,0.0,0.0,False,False
6,T1_SRC_0007,T1,NODALSTACK_T1_T1_2m_refraction_F3051_x0059.0m,geode_linked,2,secondary_geode_extension,59.0,1,1,59.0,59.0,0.0,0.0,False,False
7,T1_SRC_0008,T1,NODALSTACK_T1_T1_2m_refraction_F3052_x0063.0m,geode_linked,2,secondary_geode_extension,63.0,1,1,63.0,63.0,0.0,0.0,False,False
8,T1_SRC_0009,T1,NODALSTACK_T1_T1_2m_refraction_F3053_x0067.0m,geode_linked,2,secondary_geode_extension,67.0,1,1,67.0,67.0,0.0,0.0,False,False
9,T1_SRC_0010,T1,NODALSTACK_T1_T1_2m_refraction_F3054_x0071.0m,geode_linked,2,secondary_geode_extension,71.0,1,1,71.0,71.0,0.0,0.0,False,False


## 4. Validate cluster geometry

In [4]:
cluster_issues = []

too_far = cluster_membership.loc[
    cluster_membership.source_offset_from_canonical_m.abs()
    > SOURCE_TOLERANCE_M + 1e-12
]
if len(too_far):
    cluster_issues.append(
        f'{len(too_far)} members exceed the source tolerance from their canonical source.'
    )

duplicate_membership = cluster_membership.loc[
    cluster_membership.stack_id.duplicated(keep=False)
]
if len(duplicate_membership):
    cluster_issues.append(
        f'{duplicate_membership.stack_id.nunique()} stacks appear in multiple clusters.'
    )

bad_canonical_count = (
    cluster_membership.groupby('source_cluster_id')
    .is_canonical_stack.sum()
)
bad_canonical_count = bad_canonical_count.loc[bad_canonical_count.ne(1)]
if len(bad_canonical_count):
    cluster_issues.append(
        f'{len(bad_canonical_count)} clusters do not have exactly one canonical stack.'
    )

print('Cluster integrity issues:', len(cluster_issues))
for item in cluster_issues:
    print(' -', item)

if len(too_far):
    display(too_far)
if len(duplicate_membership):
    display(duplicate_membership)
if len(bad_canonical_count):
    display(bad_canonical_count.reset_index(name='n_canonical_stacks'))

if cluster_issues:
    raise RuntimeError('Source-cluster integrity checks failed.')
else:
    print('PASS: source clusters satisfy the configured geometry rules.')

Cluster integrity issues: 0
PASS: source clusters satisfy the configured geometry rules.


## 5. Generate all pairwise comparison tasks within clusters

In [5]:
task_rows = []

for cluster_id, members in cluster_membership.groupby(
    'source_cluster_id', sort=False
):
    if len(members) < 2:
        continue

    members = members.sort_values(
        ['priority', 'source_x_m', 'stack_id'],
        kind='stable',
    ).reset_index(drop=True)

    for left_index, right_index in combinations(range(len(members)), 2):
        left = members.iloc[left_index]
        right = members.iloc[right_index]

        if left.priority < right.priority:
            preferred_reference = left
        elif right.priority < left.priority:
            preferred_reference = right
        else:
            preferred_reference = (
                left if (left.source_x_m, left.stack_id)
                <= (right.source_x_m, right.stack_id)
                else right
            )

        task_rows.append({
            'comparison_id': (
                f"{cluster_id}__{left.stack_id}__VS__{right.stack_id}"
            ),
            'source_cluster_id': cluster_id,
            'line': left.line,
            'canonical_stack_id': left.canonical_stack_id,
            'canonical_source_x_m': left.canonical_source_x_m,
            'left_stack_id': left.stack_id,
            'left_catalog_branch': left.catalog_branch,
            'left_priority': int(left.priority),
            'left_merge_stage': left.merge_stage,
            'left_source_x_m': float(left.source_x_m),
            'right_stack_id': right.stack_id,
            'right_catalog_branch': right.catalog_branch,
            'right_priority': int(right.priority),
            'right_merge_stage': right.merge_stage,
            'right_source_x_m': float(right.source_x_m),
            'source_distance_m': abs(
                float(left.source_x_m) - float(right.source_x_m)
            ),
            'is_cross_branch_comparison': (
                left.catalog_branch != right.catalog_branch
            ),
            'preferred_reference_stack_id': preferred_reference.stack_id,
            'preferred_reference_priority': int(preferred_reference.priority),
        })

comparison_tasks = pd.DataFrame(task_rows)

if len(comparison_tasks):
    comparison_tasks = comparison_tasks.sort_values(
        [
            'line', 'canonical_source_x_m', 'source_cluster_id',
            'left_priority', 'right_priority',
            'left_stack_id', 'right_stack_id',
        ]
    ).reset_index(drop=True)

print('Comparison tasks:', len(comparison_tasks))
if len(comparison_tasks):
    display(
        comparison_tasks.groupby(
            [
                'left_catalog_branch', 'right_catalog_branch',
                'is_cross_branch_comparison',
            ],
            dropna=False,
        ).size().reset_index(name='n_tasks')
    )
    display(comparison_tasks.head(30))
else:
    print('No source cluster contains more than one stack product.')

Comparison tasks: 43


,left_catalog_branch,right_catalog_branch,is_cross_branch_comparison,n_tasks
0,geode_linked,geode_linked,False,19
1,geode_linked,nodal_only,True,21
2,nodal_only,nodal_only,False,3


,comparison_id,source_cluster_id,line,canonical_stack_id,canonical_source_x_m,left_stack_id,left_catalog_branch,left_priority,left_merge_stage,left_source_x_m,right_stack_id,right_catalog_branch,right_priority,right_merge_stage,right_source_x_m,source_distance_m,is_cross_branch_comparison,preferred_reference_stack_id,preferred_reference_priority
0,T1_SRC_0022__NODALSTACK_T1_T1_1m_refraction_F3...,T1_SRC_0022,T1,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,94.5,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,geode_linked,1,reference,94.5,NODALSTACK_T1_T1_streamer_masw_F1006_x0094.5m,geode_linked,3,streamer_extension,94.5,0.0,False,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,1
1,T1_SRC_0028__NODALSTACK_T1_T1_2m_refraction_F3...,T1_SRC_0028,T1,NODALSTACK_T1_T1_2m_refraction_F3061_x0099.0m,99.0,NODALSTACK_T1_T1_2m_refraction_F3061_x0099.0m,geode_linked,2,secondary_geode_extension,99.0,NODALSTACK_T1_T1_streamer_masw_F1009_x0099.0m,geode_linked,3,streamer_extension,99.0,0.0,False,NODALSTACK_T1_T1_2m_refraction_F3061_x0099.0m,2
2,T1_SRC_0029__NODALSTACK_T1_T1_1m_refraction_F3...,T1_SRC_0029,T1,NODALSTACK_T1_T1_1m_refraction_F3014_x0100.5m,100.5,NODALSTACK_T1_T1_1m_refraction_F3014_x0100.5m,geode_linked,1,reference,100.5,NODALSTACK_T1_T1_streamer_masw_F1010_x0100.5m,geode_linked,3,streamer_extension,100.5,0.0,False,NODALSTACK_T1_T1_1m_refraction_F3014_x0100.5m,1
3,T1_SRC_0036__NODALSTACK_T1_T1_streamer_masw_F1...,T1_SRC_0036,T1,NODALSTACK_T1_T1_streamer_masw_F1013_x0105.0m,105.0,NODALSTACK_T1_T1_streamer_masw_F1013_x0105.0m,geode_linked,3,streamer_extension,105.0,NODALONLYSTACK_MAY17_105M_x0105.0m,nodal_only,4,candidate_extension,105.0,0.0,True,NODALSTACK_T1_T1_streamer_masw_F1013_x0105.0m,3
4,T1_SRC_0038__NODALSTACK_T1_T1_1m_refraction_F3...,T1_SRC_0038,T1,NODALSTACK_T1_T1_1m_refraction_F3018_x0106.5m,106.5,NODALSTACK_T1_T1_1m_refraction_F3018_x0106.5m,geode_linked,1,reference,106.5,NODALSTACK_T1_T1_streamer_masw_F1014_x0106.5m,geode_linked,3,streamer_extension,106.5,0.0,False,NODALSTACK_T1_T1_1m_refraction_F3018_x0106.5m,1
5,T1_SRC_0039__NODALSTACK_T1_T1_2m_refraction_F3...,T1_SRC_0039,T1,NODALSTACK_T1_T1_2m_refraction_F3063_x0107.0m,107.0,NODALSTACK_T1_T1_2m_refraction_F3063_x0107.0m,geode_linked,2,secondary_geode_extension,107.0,NODALONLYSTACK_MAY17_107M_x0107.0m,nodal_only,4,candidate_extension,107.0,0.0,True,NODALSTACK_T1_T1_2m_refraction_F3063_x0107.0m,2
6,T1_SRC_0040__NODALSTACK_T1_T1_streamer_masw_F1...,T1_SRC_0040,T1,NODALSTACK_T1_T1_streamer_masw_F1015_x0108.0m,108.0,NODALSTACK_T1_T1_streamer_masw_F1015_x0108.0m,geode_linked,3,streamer_extension,108.0,NODALONLYSTACK_MAY17_108M_x0108.0m,nodal_only,4,candidate_extension,108.0,0.0,True,NODALSTACK_T1_T1_streamer_masw_F1015_x0108.0m,3
7,T1_SRC_0046__NODALSTACK_T1_T1_2m_refraction_F3...,T1_SRC_0046,T1,NODALSTACK_T1_T1_2m_refraction_F3064_x0111.0m,111.0,NODALSTACK_T1_T1_2m_refraction_F3064_x0111.0m,geode_linked,2,secondary_geode_extension,111.0,NODALSTACK_T1_T1_streamer_masw_F1017_x0111.0m,geode_linked,3,streamer_extension,111.0,0.0,False,NODALSTACK_T1_T1_2m_refraction_F3064_x0111.0m,2
8,T1_SRC_0046__NODALSTACK_T1_T1_2m_refraction_F3...,T1_SRC_0046,T1,NODALSTACK_T1_T1_2m_refraction_F3064_x0111.0m,111.0,NODALSTACK_T1_T1_2m_refraction_F3064_x0111.0m,geode_linked,2,secondary_geode_extension,111.0,NODALONLYSTACK_MAY17_111M_x0111.0m,nodal_only,4,candidate_extension,111.0,0.0,True,NODALSTACK_T1_T1_2m_refraction_F3064_x0111.0m,2
9,T1_SRC_0046__NODALSTACK_T1_T1_streamer_masw_F1...,T1_SRC_0046,T1,NODALSTACK_T1_T1_2m_refraction_F3064_x0111.0m,111.0,NODALSTACK_T1_T1_streamer_masw_F1017_x0111.0m,geode_linked,3,streamer_extension,111.0,NODALONLYSTACK_MAY17_111M_x0111.0m,nodal_only,4,candidate_extension,111.0,0.0,True,NODALSTACK_T1_T1_streamer_masw_F1017_x0111.0m,3


## 6. Resolve waveform products

In [6]:
def resolve_mseed_path(stack_id, component):
    rows = files.loc[
        files.stack_id.astype(str).eq(str(stack_id))
        & files.component.eq(component)
        & files.file_type.eq('mseed')
        & files.file_exists
    ].copy()

    if rows.empty:
        return None

    if len(rows) > 1:
        rows = rows.sort_values('file_path', kind='stable')

    return str(rows.iloc[0].file_path)


if len(comparison_tasks):
    comparison_tasks['left_mseed_path'] = comparison_tasks[
        'left_stack_id'
    ].map(lambda stack_id: resolve_mseed_path(stack_id, COMPONENT))

    comparison_tasks['right_mseed_path'] = comparison_tasks[
        'right_stack_id'
    ].map(lambda stack_id: resolve_mseed_path(stack_id, COMPONENT))

    comparison_tasks['waveforms_available'] = (
        comparison_tasks.left_mseed_path.notna()
        & comparison_tasks.right_mseed_path.notna()
    )

    print(
        'Tasks with both waveform files:',
        int(comparison_tasks.waveforms_available.sum()),
        '/',
        len(comparison_tasks),
    )

    unavailable = comparison_tasks.loc[
        ~comparison_tasks.waveforms_available
    ]
    if len(unavailable):
        display(unavailable[
            ['comparison_id', 'left_mseed_path', 'right_mseed_path']
        ])

Tasks with both waveform files: 43 / 43


## 7. Waveform and receiver-matching helpers

In [7]:
def normalize_component(value):
    text = str(value).strip().upper()
    return text[-1] if text and text[-1] in 'ZNE' else text


def receiver_x_m(trace):
    value = pd.to_numeric(
        getattr(trace.stats, 'receiver_x_m', np.nan),
        errors='coerce',
    )
    if pd.notna(value):
        return float(value)

    try:
        return int(str(trace.stats.station)) / 100.0
    except Exception:
        return np.nan


def prepare_stream(path, component):
    stream = read(str(path))
    prepared = []

    for trace in stream:
        if normalize_component(trace.stats.channel) != component:
            continue
        x = receiver_x_m(trace)
        if np.isfinite(x):
            prepared.append((x, trace.copy()))

    return sorted(prepared, key=lambda item: item[0])


def one_to_one_receiver_matches(left, right, tolerance_m):
    candidates = []
    for left_index, (left_x, _) in enumerate(left):
        for right_index, (right_x, _) in enumerate(right):
            distance = abs(float(left_x) - float(right_x))
            if distance <= tolerance_m:
                candidates.append((distance, left_index, right_index))

    candidates.sort()
    used_left = set()
    used_right = set()
    matches = []

    for distance, left_index, right_index in candidates:
        if left_index in used_left or right_index in used_right:
            continue
        used_left.add(left_index)
        used_right.add(right_index)
        matches.append((left_index, right_index, distance))

    return matches


def preprocess_trace(trace):
    out = trace.copy()
    out.detrend('demean')
    out.detrend('linear')
    out.taper(max_percentage=TAPER_FRACTION, type='cosine')

    if FILTER_FREQMIN_HZ is not None and FILTER_FREQMAX_HZ is not None:
        nyquist = 0.5 * float(out.stats.sampling_rate)
        freqmax = min(FILTER_FREQMAX_HZ, 0.95 * nyquist)
        if FILTER_FREQMIN_HZ >= freqmax:
            raise ValueError(
                f'Invalid bandpass for sampling rate {out.stats.sampling_rate}: '
                f'{FILTER_FREQMIN_HZ}-{freqmax} Hz'
            )
        out.filter(
            'bandpass',
            freqmin=FILTER_FREQMIN_HZ,
            freqmax=freqmax,
            corners=FILTER_CORNERS,
            zerophase=FILTER_ZEROPHASE,
        )
    elif FILTER_FREQMIN_HZ is not None:
        out.filter(
            'highpass',
            freq=FILTER_FREQMIN_HZ,
            corners=FILTER_CORNERS,
            zerophase=FILTER_ZEROPHASE,
        )
    elif FILTER_FREQMAX_HZ is not None:
        nyquist = 0.5 * float(out.stats.sampling_rate)
        out.filter(
            'lowpass',
            freq=min(FILTER_FREQMAX_HZ, 0.95 * nyquist),
            corners=FILTER_CORNERS,
            zerophase=FILTER_ZEROPHASE,
        )

    return out


def relative_window_data(trace):
    processed = preprocess_trace(trace)
    sampling_rate = float(processed.stats.sampling_rate)

    start_index = max(0, int(round(ANALYSIS_START_S * sampling_rate)))
    if ANALYSIS_END_S is None:
        end_index = processed.stats.npts
    else:
        end_index = min(
            processed.stats.npts,
            int(round(ANALYSIS_END_S * sampling_rate)),
        )

    if end_index <= start_index:
        raise ValueError('Analysis window contains no samples.')

    data = np.asarray(processed.data[start_index:end_index], dtype=float)
    duration_s = len(data) / sampling_rate

    return {
        'trace': processed,
        'data': data,
        'sampling_rate_hz': sampling_rate,
        'analysis_start_s': start_index / sampling_rate,
        'analysis_end_s': end_index / sampling_rate,
        'analysis_duration_s': duration_s,
    }


def estimate_snr(processed_trace, signal_data):
    sampling_rate = float(processed_trace.stats.sampling_rate)
    signal_start_index = max(
        0,
        int(round(ANALYSIS_START_S * sampling_rate)),
    )
    noise_npts = max(1, int(round(NOISE_WINDOW_DURATION_S * sampling_rate)))
    noise_end = signal_start_index
    noise_start = max(0, noise_end - noise_npts)

    noise = np.asarray(
        processed_trace.data[noise_start:noise_end],
        dtype=float,
    )

    if len(noise) < 3:
        return np.nan

    signal_rms = float(np.sqrt(np.mean(np.square(signal_data))))
    noise_rms = float(np.sqrt(np.mean(np.square(noise))))

    if noise_rms <= 0:
        return np.inf if signal_rms > 0 else np.nan

    return signal_rms / noise_rms



def snr_passes(value):
    if pd.isna(value):
        return ALLOW_MISSING_SNR
    return float(value) >= MIN_SNR


def resample_arrays(left, right):
    target_rate = min(
        left['sampling_rate_hz'],
        right['sampling_rate_hz'],
    )

    left_trace = left['trace'].copy()
    right_trace = right['trace'].copy()

    if not np.isclose(left_trace.stats.sampling_rate, target_rate):
        left_trace.resample(target_rate)
    if not np.isclose(right_trace.stats.sampling_rate, target_rate):
        right_trace.resample(target_rate)

    left_window = relative_window_data(left_trace)
    right_window = relative_window_data(right_trace)

    npts = min(len(left_window['data']), len(right_window['data']))
    if npts < 3:
        raise ValueError('Insufficient common relative-time samples.')

    return (
        np.asarray(left_window['data'][:npts], dtype=float),
        np.asarray(right_window['data'][:npts], dtype=float),
        target_rate,
        left_window,
        right_window,
    )


def normalized_cross_correlation(left_data, right_data, max_shift_samples):
    left = np.asarray(left_data, dtype=float)
    right = np.asarray(right_data, dtype=float)

    finite = np.isfinite(left) & np.isfinite(right)
    if finite.sum() < 3:
        return None

    left = left[finite]
    right = right[finite]
    left -= np.mean(left)
    right -= np.mean(right)

    left_std = np.std(left)
    right_std = np.std(right)
    if left_std == 0 or right_std == 0:
        return None

    cc = correlate(
        left / left_std,
        right / right_std,
        max_shift_samples,
        demean=False,
        normalize='naive',
    )
    shift_samples, corrcoef = xcorr_max(cc, abs_max=False)

    return {
        'shift_samples': int(shift_samples),
        'corrcoef': float(corrcoef),
    }


def shift_and_overlap(left_data, right_data, shift_samples):
    left = np.asarray(left_data, dtype=float)
    right = np.asarray(right_data, dtype=float)

    if shift_samples > 0:
        left_aligned = left[shift_samples:]
        right_aligned = right[:-shift_samples]
    elif shift_samples < 0:
        left_aligned = left[:shift_samples]
        right_aligned = right[-shift_samples:]
    else:
        left_aligned = left
        right_aligned = right

    npts = min(len(left_aligned), len(right_aligned))
    return left_aligned[:npts], right_aligned[:npts]


def correlation_at_constrained_lag(
    left_data,
    right_data,
    sampling_rate,
    gather_lag_s,
):
    center_samples = int(round(gather_lag_s * sampling_rate))
    half_width_samples = max(
        1,
        int(round(COHERENT_LAG_HALF_WIDTH_S * sampling_rate)),
    )

    best = None
    for shift_samples in range(
        center_samples - half_width_samples,
        center_samples + half_width_samples + 1,
    ):
        left_aligned, right_aligned = shift_and_overlap(
            left_data,
            right_data,
            shift_samples,
        )
        if len(left_aligned) < 3:
            continue

        corr = np.corrcoef(left_aligned, right_aligned)[0, 1]
        if not np.isfinite(corr):
            continue

        if best is None or corr > best['corrcoef']:
            best = {
                'shift_samples': int(shift_samples),
                'corrcoef': float(corr),
            }

    return best


def compare_trace_pair_initial(left_trace, right_trace):
    left = relative_window_data(left_trace)
    right = relative_window_data(right_trace)

    left_data, right_data, target_rate, left_window, right_window = (
        resample_arrays(left, right)
    )

    duration_s = len(left_data) / target_rate
    if duration_s < MIN_ANALYSIS_DURATION_S:
        return {
            'status': 'insufficient_analysis_duration',
            'analysis_duration_s': duration_s,
        }

    left_snr = estimate_snr(left_window['trace'], left_data)
    right_snr = estimate_snr(right_window['trace'], right_data)

    # Use smoothed waveform envelopes for the broad lag search. This follows
    # the arrival packet rather than an individual oscillation cycle.
    left_envelope = np.abs(hilbert(left_data))
    right_envelope = np.abs(hilbert(right_data))

    # Light smoothing stabilizes the envelope lag while preserving packet timing.
    smoothing_samples = max(1, int(round(0.010 * target_rate)))
    if smoothing_samples > 1:
        kernel = np.ones(smoothing_samples, dtype=float) / smoothing_samples
        left_envelope = np.convolve(left_envelope, kernel, mode='same')
        right_envelope = np.convolve(right_envelope, kernel, mode='same')

    max_shift_samples = max(
        1,
        int(round(COARSE_MAX_LAG_S * target_rate)),
    )
    result = normalized_cross_correlation(
        left_envelope,
        right_envelope,
        max_shift_samples,
    )

    if result is None:
        return {
            'status': 'correlation_failed',
            'analysis_duration_s': duration_s,
            'left_snr': left_snr,
            'right_snr': right_snr,
        }

    coarse_lag_s = result['shift_samples'] / target_rate
    at_search_boundary = (
        abs(result['shift_samples']) >= max_shift_samples
    )

    return {
        'status': 'initial_ok',
        'analysis_duration_s': duration_s,
        'sampling_rate_hz': target_rate,
        'n_samples': int(len(left_data)),
        'left_snr': left_snr,
        'right_snr': right_snr,
        'coarse_lag_samples': result['shift_samples'],
        'coarse_lag_s': coarse_lag_s,
        'coarse_envelope_corrcoef': result['corrcoef'],
        'coarse_lag_at_search_boundary': at_search_boundary,
        'left_data': left_data,
        'right_data': right_data,
        'left_starttime_utc': str(left_trace.stats.starttime),
        'right_starttime_utc': str(right_trace.stats.starttime),
        'absolute_starttime_difference_s': float(
            right_trace.stats.starttime - left_trace.stats.starttime
        ),
    }


def finalize_trace_pair(initial_result, gather_lag_s):
    if initial_result.get('status') != 'initial_ok':
        return initial_result.copy()

    left_data = initial_result['left_data']
    right_data = initial_result['right_data']
    sampling_rate = initial_result['sampling_rate_hz']

    constrained = correlation_at_constrained_lag(
        left_data,
        right_data,
        sampling_rate,
        gather_lag_s,
    )

    if constrained is None:
        result = initial_result.copy()
        result['status'] = 'constrained_correlation_failed'
        return result

    left_aligned, right_aligned = shift_and_overlap(
        left_data,
        right_data,
        constrained['shift_samples'],
    )

    signed_corr = float(
        np.corrcoef(left_aligned, right_aligned)[0, 1]
    )
    absolute_corr = abs(signed_corr)

    left_envelope = np.abs(hilbert(left_aligned))
    right_envelope = np.abs(hilbert(right_aligned))
    if np.std(left_envelope) > 0 and np.std(right_envelope) > 0:
        envelope_corr = float(
            np.corrcoef(left_envelope, right_envelope)[0, 1]
        )
    else:
        envelope_corr = np.nan

    result = initial_result.copy()
    result.update({
        'status': 'ok',
        'gather_lag_s': float(gather_lag_s),
        'lag_samples': constrained['shift_samples'],
        'lag_s': constrained['shift_samples'] / sampling_rate,
        'lag_deviation_from_gather_s': (
            constrained['shift_samples'] / sampling_rate
            - gather_lag_s
        ),
        'signed_corrcoef': signed_corr,
        'absolute_corrcoef': absolute_corr,
        'envelope_corrcoef': envelope_corr,
        'polarity_sign': (
            1 if signed_corr > 0 else -1 if signed_corr < 0 else 0
        ),
    })

    # Remove arrays before exporting.
    result.pop('left_data', None)
    result.pop('right_data', None)
    return result

## 8. Compare common receivers for every task

In [8]:
trace_rows = []
pair_stream_cache = {}

for task in comparison_tasks.itertuples(index=False):
    if not task.waveforms_available:
        continue

    try:
        left_stream = prepare_stream(task.left_mseed_path, COMPONENT)
        right_stream = prepare_stream(task.right_mseed_path, COMPONENT)
        matches = one_to_one_receiver_matches(
            left_stream,
            right_stream,
            RECEIVER_TOLERANCE_M,
        )

        pair_stream_cache[task.comparison_id] = (
            left_stream,
            right_stream,
            matches,
        )

        if not matches:
            trace_rows.append({
                'comparison_id': task.comparison_id,
                'source_cluster_id': task.source_cluster_id,
                'line': task.line,
                'left_stack_id': task.left_stack_id,
                'right_stack_id': task.right_stack_id,
                'source_distance_m': task.source_distance_m,
                'component': COMPONENT,
                'status': 'no_common_receivers',
            })
            continue

        initial_rows = []

        for left_index, right_index, receiver_distance in matches:
            left_x, left_trace = left_stream[left_index]
            right_x, right_trace = right_stream[right_index]

            result = compare_trace_pair_initial(
                left_trace,
                right_trace,
            )

            initial_rows.append({
                'comparison_id': task.comparison_id,
                'source_cluster_id': task.source_cluster_id,
                'line': task.line,
                'left_stack_id': task.left_stack_id,
                'right_stack_id': task.right_stack_id,
                'source_distance_m': task.source_distance_m,
                'component': COMPONENT,
                'left_receiver_x_m': left_x,
                'right_receiver_x_m': right_x,
                'receiver_distance_m': receiver_distance,
                'left_station': left_trace.stats.station,
                'right_station': right_trace.stats.station,
                **result,
            })

        initial_frame = pd.DataFrame(initial_rows)

        valid_snr = (
            initial_frame.left_snr.map(snr_passes)
            & initial_frame.right_snr.map(snr_passes)
        )

        lag_candidates = initial_frame.loc[
            initial_frame.status.eq('initial_ok')
            & initial_frame.coarse_envelope_corrcoef.ge(
                MIN_COARSE_ENVELOPE_CORRELATION_FOR_LAG
            )
            & valid_snr
        ].copy()

        if len(lag_candidates) >= MIN_RECEIVERS_FOR_GATHER_LAG:
            preliminary_lag = float(
                lag_candidates.coarse_lag_s.median()
            )
            coherent = lag_candidates.loc[
                (
                    lag_candidates.coarse_lag_s
                    - preliminary_lag
                ).abs().le(MAX_COARSE_LAG_DEVIATION_S)
            ].copy()

            if len(coherent) >= MIN_RECEIVERS_FOR_GATHER_LAG:
                gather_lag_s = float(
                    coherent.coarse_lag_s.median()
                )
                gather_lag_status = 'estimated'
            else:
                gather_lag_s = preliminary_lag
                gather_lag_status = 'weak_coherence'
        else:
            gather_lag_s = 0.0
            gather_lag_status = 'insufficient_lag_receivers'

        for row in initial_rows:
            finalized = finalize_trace_pair(
                row,
                gather_lag_s,
            )
            finalized['gather_lag_status'] = gather_lag_status
            trace_rows.append(finalized)

    except Exception as exc:
        trace_rows.append({
            'comparison_id': task.comparison_id,
            'source_cluster_id': task.source_cluster_id,
            'line': task.line,
            'left_stack_id': task.left_stack_id,
            'right_stack_id': task.right_stack_id,
            'source_distance_m': task.source_distance_m,
            'component': COMPONENT,
            'status': 'pair_processing_error',
            'error': repr(exc),
        })

trace_qc = pd.DataFrame(trace_rows)

print('Trace comparison rows:', len(trace_qc))
if len(trace_qc):
    display(
        trace_qc.groupby('status', dropna=False)
        .size().reset_index(name='n_rows')
    )


Trace comparison rows: 1212


,status,n_rows
0,ok,1212


## 9. Summarize pair-level waveform evidence

In [9]:
review_rows = []

for task in comparison_tasks.itertuples(index=False):
    rows = trace_qc.loc[
        trace_qc.comparison_id.eq(task.comparison_id)
        & trace_qc.status.eq('ok')
    ].copy()

    if len(rows):
        passing = (
            rows.signed_corrcoef.ge(TRACE_ACCEPT_CORRELATION)
            | rows.envelope_corrcoef.ge(
                TRACE_ACCEPT_ENVELOPE_CORRELATION
            )
        )
        coherent_lag = rows.lag_deviation_from_gather_s.abs().le(
            COHERENT_LAG_HALF_WIDTH_S
        )

        n_common = len(rows)
        fraction_passing = float(passing.mean())
        fraction_coherent_lag = float(coherent_lag.mean())
        fraction_positive_polarity = float(
            rows.polarity_sign.gt(0).mean()
        )

        median_coarse_envelope_corr = float(
            rows.coarse_envelope_corrcoef.median()
        )
        fraction_coarse_lags_at_boundary = float(
            rows.coarse_lag_at_search_boundary.fillna(False).mean()
        )
        median_signed_corr = float(rows.signed_corrcoef.median())
        median_absolute_corr = float(
            rows.absolute_corrcoef.median()
        )
        median_envelope_corr = float(
            rows.envelope_corrcoef.median()
        )
        minimum_signed_corr = float(rows.signed_corrcoef.min())
        gather_lag_s = float(rows.gather_lag_s.median())
        median_lag = float(rows.lag_s.median())
        lag_mad = float(
            np.median(
                np.abs(rows.lag_s - np.median(rows.lag_s))
            )
        )
        median_left_snr = float(rows.left_snr.median())
        median_right_snr = float(rows.right_snr.median())
        gather_lag_status = (
            rows.gather_lag_status.mode().iloc[0]
            if rows.gather_lag_status.notna().any()
            else 'unknown'
        )
    else:
        n_common = 0
        fraction_passing = np.nan
        fraction_coherent_lag = np.nan
        fraction_positive_polarity = np.nan
        median_coarse_envelope_corr = np.nan
        fraction_coarse_lags_at_boundary = np.nan
        median_signed_corr = np.nan
        median_absolute_corr = np.nan
        median_envelope_corr = np.nan
        minimum_signed_corr = np.nan
        gather_lag_s = np.nan
        median_lag = np.nan
        lag_mad = np.nan
        median_left_snr = np.nan
        median_right_snr = np.nan
        gather_lag_status = 'unavailable'

    enough_receivers = (
        n_common >= MIN_COMMON_RECEIVERS_FOR_GATHER
    )
    good_snr = (
        snr_passes(median_left_snr)
        and snr_passes(median_right_snr)
    )
    passes_waveform_thresholds = (
        enough_receivers
        and good_snr
        and (
            median_signed_corr >= GATHER_ACCEPT_MEDIAN_CORRELATION
            or median_envelope_corr
            >= GATHER_ACCEPT_MEDIAN_ENVELOPE_CORRELATION
        )
        and fraction_passing >= GATHER_ACCEPT_FRACTION
        and fraction_coherent_lag
        >= GATHER_ACCEPT_COHERENT_LAG_FRACTION
        and lag_mad <= GATHER_MAX_LAG_MAD_S
    )

    if not task.waveforms_available:
        automatic_status = 'missing_waveform_product'
    elif not enough_receivers:
        automatic_status = 'waveform_match_inconclusive'
    elif not good_snr:
        automatic_status = 'waveform_match_inconclusive'
    elif passes_waveform_thresholds:
        automatic_status = 'waveform_match_supported'
    else:
        automatic_status = 'waveform_match_inconclusive'

    row = task._asdict()
    row.update({
        'n_common_receivers': n_common,
        'n_trace_correlations_passing': (
            int(passing.sum()) if len(rows) else 0
        ),
        'fraction_trace_correlations_passing': fraction_passing,
        'fraction_receivers_with_coherent_lag': (
            fraction_coherent_lag
        ),
        'fraction_positive_polarity': (
            fraction_positive_polarity
        ),
        'median_coarse_envelope_corrcoef': median_coarse_envelope_corr,
        'fraction_coarse_lags_at_search_boundary': (
            fraction_coarse_lags_at_boundary
        ),
        'median_signed_corrcoef': median_signed_corr,
        'median_absolute_corrcoef': median_absolute_corr,
        'median_envelope_corrcoef': median_envelope_corr,
        'minimum_signed_corrcoef': minimum_signed_corr,
        'gather_lag_s': gather_lag_s,
        'median_receiver_lag_s': median_lag,
        'lag_mad_s': lag_mad,
        'median_left_snr': median_left_snr,
        'snr_available_fraction': (
            float(
                (
                    rows.left_snr.notna()
                    & rows.right_snr.notna()
                ).mean()
            ) if len(rows) else np.nan
        ),
        'median_right_snr': median_right_snr,
        'gather_lag_status': gather_lag_status,
        'automatic_status': automatic_status,
        'manual_review_status': 'not_reviewed',
        'same_physical_shot_decision': 'undecided',
        'review_notes': '',
    })
    review_rows.append(row)

pair_review = pd.DataFrame(review_rows)

if len(pair_review):
    pair_review = pair_review.sort_values(
        [
            'line', 'canonical_source_x_m',
            'source_cluster_id', 'left_priority',
            'right_priority', 'left_stack_id',
            'right_stack_id',
        ]
    ).reset_index(drop=True)

print('Pair-level review rows:', len(pair_review))
if len(pair_review):
    display(
        pair_review.groupby('automatic_status', dropna=False)
        .size().reset_index(name='n_comparisons')
    )
    display(pair_review.head(30))


Pair-level review rows: 43


,automatic_status,n_comparisons
0,waveform_match_inconclusive,1
1,waveform_match_supported,42


,comparison_id,source_cluster_id,line,canonical_stack_id,canonical_source_x_m,left_stack_id,left_catalog_branch,left_priority,left_merge_stage,left_source_x_m,right_stack_id,right_catalog_branch,right_priority,right_merge_stage,right_source_x_m,source_distance_m,is_cross_branch_comparison,preferred_reference_stack_id,preferred_reference_priority,left_mseed_path,right_mseed_path,waveforms_available,n_common_receivers,n_trace_correlations_passing,fraction_trace_correlations_passing,fraction_receivers_with_coherent_lag,fraction_positive_polarity,median_coarse_envelope_corrcoef,fraction_coarse_lags_at_search_boundary,median_signed_corrcoef,median_absolute_corrcoef,median_envelope_corrcoef,minimum_signed_corrcoef,gather_lag_s,median_receiver_lag_s,lag_mad_s,median_left_snr,snr_available_fraction,median_right_snr,gather_lag_status,automatic_status,manual_review_status,same_physical_shot_decision,review_notes
0,T1_SRC_0022__NODALSTACK_T1_T1_1m_refraction_F3...,T1_SRC_0022,T1,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,94.5,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,geode_linked,1,reference,94.5,NODALSTACK_T1_T1_streamer_masw_F1006_x0094.5m,geode_linked,3,streamer_extension,94.5,0.0,False,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,1,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,True,26,26,1.000000,1.0,1.0,0.835855,0.0,0.733448,0.733448,0.851640,0.410489,-0.283,-0.286,0.000,NaN,0.0,NaN,estimated,waveform_match_supported,not_reviewed,undecided,
1,T1_SRC_0028__NODALSTACK_T1_T1_2m_refraction_F3...,T1_SRC_0028,T1,NODALSTACK_T1_T1_2m_refraction_F3061_x0099.0m,99.0,NODALSTACK_T1_T1_2m_refraction_F3061_x0099.0m,geode_linked,2,secondary_geode_extension,99.0,NODALSTACK_T1_T1_streamer_masw_F1009_x0099.0m,geode_linked,3,streamer_extension,99.0,0.0,False,NODALSTACK_T1_T1_2m_refraction_F3061_x0099.0m,2,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,True,26,26,1.000000,1.0,1.0,0.937306,0.0,0.835235,0.835235,0.914939,0.589873,0.014,0.014,0.000,NaN,0.0,NaN,estimated,waveform_match_supported,not_reviewed,undecided,
2,T1_SRC_0029__NODALSTACK_T1_T1_1m_refraction_F3...,T1_SRC_0029,T1,NODALSTACK_T1_T1_1m_refraction_F3014_x0100.5m,100.5,NODALSTACK_T1_T1_1m_refraction_F3014_x0100.5m,geode_linked,1,reference,100.5,NODALSTACK_T1_T1_streamer_masw_F1010_x0100.5m,geode_linked,3,streamer_extension,100.5,0.0,False,NODALSTACK_T1_T1_1m_refraction_F3014_x0100.5m,1,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,True,26,23,0.884615,1.0,1.0,0.886680,0.0,0.771036,0.771036,0.861598,0.308585,0.204,0.204,0.001,NaN,0.0,NaN,estimated,waveform_match_supported,not_reviewed,undecided,
3,T1_SRC_0036__NODALSTACK_T1_T1_streamer_masw_F1...,T1_SRC_0036,T1,NODALSTACK_T1_T1_streamer_masw_F1013_x0105.0m,105.0,NODALSTACK_T1_T1_streamer_masw_F1013_x0105.0m,geode_linked,3,streamer_extension,105.0,NODALONLYSTACK_MAY17_105M_x0105.0m,nodal_only,4,candidate_extension,105.0,0.0,True,NODALSTACK_T1_T1_streamer_masw_F1013_x0105.0m,3,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,/Volumes/tachyon/LBSSP_DATA/nodal_only_stacked...,True,26,25,0.961538,1.0,1.0,0.919699,0.0,0.818349,0.818349,0.895906,0.438388,-0.064,-0.062,0.004,NaN,0.0,NaN,estimated,waveform_match_supported,not_reviewed,undecided,
4,T1_SRC_0038__NODALSTACK_T1_T1_1m_refraction_F3...,T1_SRC_0038,T1,NODALSTACK_T1_T1_1m_refraction_F3018_x0106.5m,106.5,NODALSTACK_T1_T1_1m_refraction_F3018_x0106.5m,geode_linked,1,reference,106.5,NODALSTACK_T1_T1_streamer_masw_F1014_x0106.5m,geode_linked,3,streamer_extension,106.5,0.0,False,NODALSTACK_T1_T1_1m_refraction_F3018_x0106.5m,1,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,True,26,26,1.000000,1.0,1.0,0.908852,0.0,0.786001,0.786001,0.907654,0.635933,0.124,0.120,0.000,NaN,0.0,NaN,estimated,waveform_match_supported,not_reviewed,undecided,
5,T1_SRC_0039__NODALSTACK_T1_T1_2m_refractio

## 10. Build cluster-level review summary

In [10]:
cluster_review = source_clusters.copy()

if len(pair_review):
    pair_counts = (
        pair_review.groupby('source_cluster_id')
        .agg(
            n_comparison_tasks=('comparison_id', 'size'),
            n_waveform_match_supported=(
                'automatic_status',
                lambda series: int(
                    series.eq('waveform_match_supported').sum()
                ),
            ),
            n_waveform_match_inconclusive=(
                'automatic_status',
                lambda series: int(
                    series.eq('waveform_match_inconclusive').sum()
                ),
            ),
            n_insufficient_common_receivers=(
                'automatic_status',
                lambda series: int(
                    series.eq('insufficient_common_receivers').sum()
                ),
            ),
            best_median_trace_corrcoef=(
                'median_signed_corrcoef', 'max'
            ),
            worst_median_trace_corrcoef=(
                'median_signed_corrcoef', 'min'
            ),
        )
        .reset_index()
    )

    cluster_review = cluster_review.merge(
        pair_counts,
        on='source_cluster_id',
        how='left',
        validate='one_to_one',
    )
else:
    cluster_review['n_comparison_tasks'] = 0

for column in [
    'n_comparison_tasks',
    'n_waveform_match_supported',
    'n_waveform_match_inconclusive',
    'n_insufficient_common_receivers',
]:
    if column not in cluster_review:
        cluster_review[column] = 0
    cluster_review[column] = cluster_review[column].fillna(0).astype(int)

cluster_review['manual_cluster_status'] = 'not_reviewed'
cluster_review['canonical_source_decision'] = np.where(
    cluster_review.n_stack_products.eq(1),
    'single_stack_only',
    'undecided',
)
cluster_review['cluster_review_notes'] = ''

display(cluster_review.head(30))

,source_cluster_id,line,canonical_stack_id,canonical_catalog_branch,canonical_priority,canonical_merge_stage,canonical_source_x_m,n_stack_products,n_catalog_branches,minimum_source_x_m,maximum_source_x_m,source_span_m,maximum_abs_offset_from_canonical_m,is_multi_stack_cluster,is_cross_branch_cluster,n_comparison_tasks,n_waveform_match_supported,n_waveform_match_inconclusive,n_insufficient_common_receivers,best_median_trace_corrcoef,worst_median_trace_corrcoef,manual_cluster_status,canonical_source_decision,cluster_review_notes
0,T1_SRC_0001,T1,NODALONLYSTACK_MAY19_010M_x0010.0m,nodal_only,4,candidate_extension,10.0,1,1,10.0,10.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,
1,T1_SRC_0002,T1,NODALONLYSTACK_MAY19_036M_x0036.0m,nodal_only,4,candidate_extension,36.0,1,1,36.0,36.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,
2,T1_SRC_0003,T1,NODALSTACK_T1_T1_2m_refraction_F3047_x0043.0m,geode_linked,2,secondary_geode_extension,43.0,1,1,43.0,43.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,
3,T1_SRC_0004,T1,NODALSTACK_T1_T1_2m_refraction_F3048_x0047.0m,geode_linked,2,secondary_geode_extension,47.0,1,1,47.0,47.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,
4,T1_SRC_0005,T1,NODALSTACK_T1_T1_2m_refraction_F3049_x0051.0m,geode_linked,2,secondary_geode_extension,51.0,1,1,51.0,51.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,
5,T1_SRC_0006,T1,NODALSTACK_T1_T1_2m_refraction_F3050_x0055.0m,geode_linked,2,secondary_geode_extension,55.0,1,1,55.0,55.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,
6,T1_SRC_0007,T1,NODALSTACK_T1_T1_2m_refraction_F3051_x0059.0m,geode_linked,2,secondary_geode_extension,59.0,1,1,59.0,59.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,
7,T1_SRC_0008,T1,NODALSTACK_T1_T1_2m_refraction_F3052_x0063.0m,geode_linked,2,secondary_geode_extension,63.0,1,1,63.0,63.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,
8,T1_SRC_0009,T1,NODALSTACK_T1_T1_2m_refraction_F3053_x0067.0m,geode_linked,2,secondary_geode_extension,67.0,1,1,67.0,67.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,
9,T1_SRC_0010,T1,NODALSTACK_T1_T1_2m_refraction_F3054_x0071.0m,geode_linked,2,secondary_geode_extension,71.0,1,1,71.0,71.0,0.0,0.0,False,False,0,0,0,0,NaN,NaN,not_reviewed,single_stack_only,


## 11. Create review figures

In [11]:
def safe_name(value):
    return ''.join(
        character if character.isalnum() or character in '-_.'
        else '_'
        for character in str(value)
    )


figure_paths = []
figure_reviews = pair_review.copy()

if MAX_FIGURES is not None:
    figure_reviews = figure_reviews.head(MAX_FIGURES)

if MAKE_REVIEW_FIGURES:
    for review in figure_reviews.itertuples(index=False):
        rows = trace_qc.loc[
            trace_qc.comparison_id.eq(review.comparison_id)
            & trace_qc.status.eq('ok')
        ].copy()

        if rows.empty or review.comparison_id not in pair_stream_cache:
            continue

        left_stream, right_stream, _ = pair_stream_cache[
            review.comparison_id
        ]

        fig, ax = plt.subplots(figsize=(12, 8))
        vertical_offset = 0.0

        for row in rows.sort_values(
            'left_receiver_x_m'
        ).itertuples(index=False):
            left_item = min(
                left_stream,
                key=lambda item: abs(
                    item[0] - row.left_receiver_x_m
                ),
            )
            right_item = min(
                right_stream,
                key=lambda item: abs(
                    item[0] - row.right_receiver_x_m
                ),
            )

            left = relative_window_data(left_item[1])
            right = relative_window_data(right_item[1])
            left_data, right_data, rate, _, _ = resample_arrays(
                left,
                right,
            )
            left_aligned, right_aligned = shift_and_overlap(
                left_data,
                right_data,
                int(row.lag_samples),
            )

            if len(left_aligned) < 3:
                continue

            left_scale = np.nanmax(np.abs(left_aligned))
            right_scale = np.nanmax(np.abs(right_aligned))
            if np.isfinite(left_scale) and left_scale > 0:
                left_aligned = left_aligned / left_scale
            if np.isfinite(right_scale) and right_scale > 0:
                right_aligned = right_aligned / right_scale

            times = (
                ANALYSIS_START_S
                + np.arange(len(left_aligned)) / rate
            )
            ax.plot(
                times,
                left_aligned + vertical_offset,
                linewidth=0.8,
            )
            ax.plot(
                times,
                right_aligned + vertical_offset,
                linewidth=0.8,
                alpha=0.75,
            )
            ax.text(
                times[-1] if len(times) else ANALYSIS_START_S,
                vertical_offset,
                (
                    f" x={row.left_receiver_x_m:.2f}/"
                    f"{row.right_receiver_x_m:.2f} m, "
                    f"r={row.signed_corrcoef:.2f}, "
                    f"env={row.envelope_corrcoef:.2f}, "
                    f"lag={row.lag_s * 1000:.1f} ms"
                ),
                va='center',
                fontsize=7,
            )
            vertical_offset += 2.5

        ax.set_title(
            f"{review.source_cluster_id}: "
            f"{review.left_stack_id} vs {review.right_stack_id}\n"
            f"source separation={review.source_distance_m:.3f} m; "
            f"median r={review.median_signed_corrcoef:.3f}; "
            f"coarse lag={review.gather_lag_s:.3f} s; "
            f"median envelope r={review.median_envelope_corrcoef:.3f}; "
            f"status={review.automatic_status}"
        )
        ax.set_xlabel('Relative time in analysis window (s)')
        ax.set_ylabel(
            'Lag-aligned normalized traces with vertical offsets'
        )
        ax.grid(True, alpha=0.25)
        fig.tight_layout()

        figure_path = (
            FIGURE_ROOT
            / f"{safe_name(review.comparison_id)}_{COMPONENT}.png"
        )
        fig.savefig(figure_path, dpi=180)
        plt.close(fig)
        figure_paths.append(str(figure_path))

print('Review figures written:', len(figure_paths))


Review figures written: 43


## 12. Export cluster, task, and waveform-review tables

In [12]:
OUTPUTS = {
    'clusters': OUT_ROOT / '97_source_clusters.csv',
    'membership': OUT_ROOT / '97_source_cluster_membership.csv',
    'tasks': OUT_ROOT / '97_source_cluster_comparison_tasks.csv',
    'trace_qc': OUT_ROOT / '97_candidate_trace_correlation_qc.csv',
    'pair_review': OUT_ROOT / '97_candidate_pair_review.csv',
    'cluster_review': OUT_ROOT / '97_source_cluster_review.csv',
    'summary': OUT_ROOT / '97_source_cluster_review_summary.csv',
}

source_clusters.to_csv(OUTPUTS['clusters'], index=False)
cluster_membership.to_csv(OUTPUTS['membership'], index=False)
comparison_tasks.to_csv(OUTPUTS['tasks'], index=False)
trace_qc.to_csv(OUTPUTS['trace_qc'], index=False)
pair_review.to_csv(OUTPUTS['pair_review'], index=False)
cluster_review.to_csv(OUTPUTS['cluster_review'], index=False)

summary = pd.DataFrame([
    ('source_tolerance_m', SOURCE_TOLERANCE_M),
    ('receiver_tolerance_m', RECEIVER_TOLERANCE_M),
    ('component', COMPONENT),
    ('input_stack_products', len(stacks)),
    ('source_clusters', len(source_clusters)),
    ('single_stack_clusters', int(
        source_clusters.n_stack_products.eq(1).sum()
    ) if len(source_clusters) else 0),
    ('multi_stack_clusters', int(
        source_clusters.n_stack_products.gt(1).sum()
    ) if len(source_clusters) else 0),
    ('cross_branch_clusters', int(
        source_clusters.is_cross_branch_cluster.sum()
    ) if len(source_clusters) else 0),
    ('comparison_tasks', len(comparison_tasks)),
    ('tasks_with_waveforms', int(
        comparison_tasks.waveforms_available.sum()
    ) if len(comparison_tasks) else 0),
    ('successful_trace_comparisons', int(
        trace_qc.status.eq('ok').sum()
    ) if len(trace_qc) else 0),
    ('waveform_match_supported', int(
        pair_review.automatic_status.eq(
            'waveform_match_supported'
        ).sum()
    ) if len(pair_review) else 0),
    ('waveform_match_inconclusive', int(
        pair_review.automatic_status.eq(
            'waveform_match_inconclusive'
        ).sum()
    ) if len(pair_review) else 0),
    ('review_figures', len(figure_paths)),
], columns=['metric', 'value'])

summary.to_csv(OUTPUTS['summary'], index=False)
display(summary)

print('\nWritten:')
for key, path in OUTPUTS.items():
    print(f'  {key:16s} {path}')
print('  figures          ', FIGURE_ROOT)

,metric,value
0,source_tolerance_m,0.25
1,receiver_tolerance_m,0.25
2,component,Z
3,input_stack_products,238
4,source_clusters,198
5,single_stack_clusters,161
6,multi_stack_clusters,37
7,cross_branch_clusters,18
8,comparison_tasks,43
9,tasks_with_waveforms,43



Written:
  clusters         /Volumes/tachyon/LBSSP_DATA/97_nodal_source_cluster_review/97_source_clusters.csv
  membership       /Volumes/tachyon/LBSSP_DATA/97_nodal_source_cluster_review/97_source_cluster_membership.csv
  tasks            /Volumes/tachyon/LBSSP_DATA/97_nodal_source_cluster_review/97_source_cluster_comparison_tasks.csv
  trace_qc         /Volumes/tachyon/LBSSP_DATA/97_nodal_source_cluster_review/97_candidate_trace_correlation_qc.csv
  pair_review      /Volumes/tachyon/LBSSP_DATA/97_nodal_source_cluster_review/97_candidate_pair_review.csv
  cluster_review   /Volumes/tachyon/LBSSP_DATA/97_nodal_source_cluster_review/97_source_cluster_review.csv
  summary          /Volumes/tachyon/LBSSP_DATA/97_nodal_source_cluster_review/97_source_cluster_review_summary.csv
  figures           /Volumes/tachyon/LBSSP_DATA/97_nodal_source_cluster_review/figures


## 13. Interpretation and later integration

The source cluster is a geometric hypothesis. Waveform evidence may strongly
support a pairing, but a weak or inconsistent waveform result is now classified
as `waveform_match_inconclusive`, not as a rejection.

Review:

- `97_candidate_pair_review.csv`
- `97_source_cluster_review.csv`
- the generated lag-aligned figures

Useful diagnostics include:

- `gather_lag_s` and `median_coarse_envelope_corrcoef`
- `fraction_coarse_lags_at_search_boundary`
- `median_signed_corrcoef`
- `median_absolute_corrcoef`
- `median_envelope_corrcoef`
- `gather_lag_s`
- `lag_mad_s`
- `fraction_receivers_with_coherent_lag`
- `fraction_positive_polarity`
- median SNR values

Update the manual-decision columns only after examining both geometry and
waveform diagnostics.

A later integration notebook should merge only approved relationships, retain
every original stack identifier, preserve receiver-level provenance, and use the
canonical stack/source defined by the lowest numerical priority.

A nonzero `fraction_coarse_lags_at_search_boundary` indicates that the configured ±coarse-lag range may still be too narrow.


When `ANALYSIS_START_S = 0`, a pre-signal noise window may not exist. In that
case SNR is recorded as unavailable and does not block coarse-lag estimation
when `ALLOW_MISSING_SNR = True`. The exported `snr_available_fraction` shows how
much of each comparison had a valid SNR estimate.
